# Predicting SLA Breaches at Ticket Creation: An Exploratory Analysis of ITSM Incidents

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv(r'C:\Users\Vishnu Ravi\python files\IT ticket data analysis project\incident_event_log.csv')
pd.set_option('display.max_columns', None)

## Problem Definition

**What is being predicted?**
Binary classification: SLA breach (made_sla == False)

**When is the prediction made?**
At ticket creation

**Why does it matter?**
Prioritization, staffing, escalation

**Real-world usage:**
This model could be used to flag high-risk tickets for early intervention.

Answer explicitly:

What is being predicted?
Binary classification: SLA breach (made_sla == False)

When is the prediction made?
At ticket creation

Why does it matter?
Prioritization, staffing, escalation

**Include a 1-paragraph “real-world usage”:**

“This model could be used to flag high-risk tickets for early intervention.”

## 2. Data Understanding and Granularity

## 2.1 Dataset Overview

This analysis uses the IT Incident Event Log dataset, a publicly available dataset commonly used to study service desk operations and SLA performance. The raw data contains 119,998 rows and 36 columns, representing logged events associated with IT incident tickets.

Each incident is identified by a unique number (20,769 distinct incidents). Rather than being incident-level, the dataset is event-level, meaning that each row corresponds to a logged update or state change for an incident over its lifecycle.

Key implications of this structure:

A single incident (number) appears in multiple rows.

Certain attributes (e.g., incident_state, sys_mod_count, sys_updated_at) evolve over time.

Time-based fields such as resolved_at and closed_at may appear populated even in early lifecycle rows due to system backfilling behavior.

Understanding and properly handling this granularity is critical to avoid incorrect aggregation, double-counting, or temporal data leakage.

In [14]:

df = pd.read_csv("incident_event_log.csv")

# Basic shape and structure
print("Raw dataset shape:", df.shape)
n_incidents = df['number'].nunique()
print(f"number of incidents: {n_incidents}")

print("Average rows per incident:", df.shape[0] / n_incidents)
# Preview data
df.head()




Raw dataset shape: (119998, 36)
number of incidents: 20769
Average rows per incident: 5.777745678655688


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


2.2 Event-Level vs. Incident-Level Structure

An inspection of the data confirms that:

Multiple rows correspond to the same incident number.

Rows represent chronological events, not independent observations.

Incident lifecycle information is spread across multiple records.

Because the modeling goal is to predict whether an incident will miss its SLA, the appropriate unit of analysis for prediction is the incident, not the event. Therefore, all downstream analysis and modeling must be performed at the incident level, with exactly one row per incident.

To support this, the dataset will later be aggregated such that:

Each incident is represented by a single record.

Features are derived from information available at a clearly defined point in time.

The target variable (made_sla) is uniquely defined per incident.


In [27]:
# Count how many rows each incident appears in
incident_event_counts = df['number'].value_counts()

incident_event_counts.describe()
# Inspect incidents with multiple lifecycle events
incident_event_counts.head(5)
# Show an example of a single incident across multiple rows
example_incident = incident_event_counts.index[1]
df[df['number'] == example_incident].head(5)


,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
26389,INC0005927,New,True,0,0,0,True,Caller 1173,Opened by 301,11/3/2016 11:25,?,?,Updated by 908,11/3/2016 11:25,Phone,Location 81,Category 51,Subcategory 301,Symptom 592,?,2 - Medium,2 - Medium,3 - Moderate,Group 54,?,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 15,29/4/2016 09:37,4/5/2016 10:07
26390,INC0005927,New,True,0,0,1,True,Caller 1173,Opened by 301,11/3/2016 11:25,?,?,Updated by 874,11/3/2016 12:00,Phone,Location 81,Category 51,Subcategory 301,Symptom 592,?,2 - Medium,2 - Medium,3 - Moderate,Group 54,?,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 15,29/4/2016 09:37,4/5/2016 10:07
26391,INC0005927,New,True,1,0,2,True,Caller 1173,Opened by 301,11/3/2016 11:25,?,?,Updated by 874,11/3/2016 12:00,Phone,Location 81,Category 51,Subcategory 301,Symptom 592,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,?,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 15,29/4/2016 09:37,4/5/2016 10:07
26392,INC0005927,New,True,2,0,5,True,Caller 1173,Opened by 301,11/3/2016 11:25,?,?,Updated by 915,11/3/2016 13:57,Phone,Location 81,Category 51,Subcategory 301,Symptom 592,?,2 - Medium,2 - Medium,3 - Moderate,Group 54,?,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 15,29/4/2016 09:37,4/5/2016 10:07
26393,INC0005927,New,True,2,0,6,True,Caller 1173,Opened by 301,11/3/2016 11:25,?,?,Updated by 874,11/3/2016 16:24,Phone,Location 81,Category 51,Subcategory 301,Symptom 592,?,2 - Medium,2 - Medium,3 - Moderate,Group 54,?,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 15,29/4/2016 09:37,4/5/2016 10:07


2.3 Prediction Timing and Feature Availability

The prediction task is framed as a classification problem: predicting whether an incident will miss its SLA (made_sla = False) at the time the ticket is created.

This framing imposes an important constraint:

Only information available at or before ticket creation may be used for exploratory analysis and modeling.

As a result:

Post-creation attributes (e.g., resolved_at, closed_at, sys_mod_count, reassignment counts, and later incident_state transitions) must be excluded from predictive feature analysis.

These post-hoc variables may still be useful for operational or descriptive analysis, but not for predictive modeling.

This distinction ensures that any insights or models developed reflect a realistic deployment scenario and avoids temporal data leakage.
It seems that opened_at does not change and merely represents the initial time that the incident is opened

In [31]:
# Show how temporal columns change over the course of the same incident
temporal_columns = [
    'number',
    'incident_state',
    'sys_mod_count',
    'sys_created_at',
    'sys_updated_at',
    'opened_at',
    'resolved_at',
    'closed_at',
    'assigned_to',
    'assignment_group'
]

df[df['number'] == example_incident][temporal_columns].head(10)



,number,incident_state,sys_mod_count,sys_created_at,sys_updated_at,opened_at,resolved_at,closed_at,assigned_to,assignment_group
26389,INC0005927,New,0,?,11/3/2016 11:25,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 54
26390,INC0005927,New,1,?,11/3/2016 12:00,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 54
26391,INC0005927,New,2,?,11/3/2016 12:00,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 70
26392,INC0005927,New,5,?,11/3/2016 13:57,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 54
26393,INC0005927,New,6,?,11/3/2016 16:24,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 54
26394,INC0005927,New,7,?,11/3/2016 16:24,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 70
26395,INC0005927,New,8,?,11/3/2016 17:45,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 58
26396,INC0005927,New,9,?,14/3/2016 07:50,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 70
26397,INC0005927,New,10,?,14/3/2016 08:20,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 76
26398,INC0005927,New,12,?,17/3/2016 10:51,11/3/2016 11:25,29/4/2016 09:37,4/5/2016 10:07,?,Group 69


In [23]:
# Show how incident_state evolves within a single incident
df[df['number'] == example_incident][
    ['number', 'incident_state', 'sys_mod_count', 'sys_updated_at']
].head(10)

,number,incident_state,sys_mod_count,sys_updated_at
81418,INC0019396,New,0,14/4/2016 20:42
81419,INC0019396,New,1,14/4/2016 21:08
81420,INC0019396,Resolved,6,15/4/2016 09:20
81421,INC0019396,Active,7,15/4/2016 15:21
81422,INC0019396,Active,9,15/4/2016 15:21
81423,INC0019396,Active,10,15/4/2016 20:30
81424,INC0019396,Active,11,15/4/2016 20:30
81425,INC0019396,Active,12,16/4/2016 23:43
81426,INC0019396,Active,16,16/4/2016 23:43
81427,INC0019396,Active,18,20/4/2016 11:12


2.4 Missing Values and Logging Behavior

Several attributes in the dataset use the string '?' to represent unknown or unrecorded values. According to the dataset documentation, these should be treated as missing information, not literal values.

Additionally, some fields (particularly time-based fields such as resolved_at) are populated retroactively by the system, meaning their presence in early lifecycle rows does not imply that the information was known at that time.

These behaviors further reinforce the need for:

Explicit missing value normalization during preprocessing.

Careful consideration of feature availability relative to prediction timing.

Incident-level aggregation before modeling.

2.5 Summary of Implications for Analysis

From this initial data understanding step, several key decisions follow:

The raw dataset cannot be modeled directly due to its event-level structure.

Incident-level aggregation is required prior to EDA and modeling.

Predictive analysis must be restricted to creation-time information.

Later lifecycle fields are excluded to prevent data leakage.